In [1]:
import numpy as np
import datetime
import os
from netCDF4 import Dataset
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import itertools
np.float = float
import cartopy
from regrid import regrid
import tqdm
import warnings
warnings.filterwarnings("ignore")

/opt/anaconda3/envs/phd/lib/python3.8/site-packages/pyproj/crs.py:77: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method.
  return _prepare_from_string(" ".join(pjargs))


In [2]:
datapath_aux = '/Users/carmennab/Dropbox/pb_new/auxiliary/'
ease_lons = np.load(f'{datapath_aux}lon_cent.npy')
ease_lats = np.load(f'{datapath_aux}lat_cent.npy')

In [5]:
### mask out data outside of study area
region_mask = np.array(Dataset(f'{datapath_aux}NSIDC_Regions_Mask_ease.nc')['mask'])
region_mask[ease_lats<55] = 0
region_mask[ease_lats>62] = 0
region_mask[ease_lons>-76] = 0

In [6]:
smos_path = '/Users/carmennab/Dropbox/pb_new/data/SMOS/'
datapath_out = '/Users/carmennab/Dropbox/pb_new/sea_ice_thickness/SMOS/'

In [7]:
d1 = datetime.date(2010, 1, 1)
d2 = datetime.date(2022, 12, 31)
dates = [d1 + datetime.timedelta(days=x) for x in range((d2-d1).days + 1)]
str_dates = [date.strftime('%Y%m%d') for date in dates]

In [8]:
for date,str_date in zip(dates,str_dates):

    smos_filepath = f'{smos_path}{date.year}/{str_date[4:6]}/SMOS_Icethickness_v3.3_north_{str_date}.nc'
    if os.path.exists(smos_filepath) == True: 
        smos_file = Dataset(smos_filepath)
        smos_lons = np.array(smos_file['longitude'])
        smos_lats = np.array(smos_file['latitude'])
        smos_sit = np.array(smos_file['sea_ice_thickness'])[0]
        sit_unc = np.array(smos_file['ice_thickness_uncertainty'])[0]
        smos_sit[sit_unc>1]=np.nan
        smos_sit[smos_sit<0]=np.nan
        smos_sit[smos_sit>1]=np.nan
        
        smos_ease = regrid(smos_sit,smos_lons,smos_lats,ease_lons,ease_lats)

        smos_ease[~np.isin(region_mask,4)] = np.nan # remove non-HB data
        if ~np.isnan(smos_ease).all():
            filepath_out = f'{datapath_out}/{date.year}/{str_date}.nc'
            if os.path.exists(filepath_out) == True: 
                ! rm {filepath_out}
            data_vars = {'sea_ice_thickness':(['x','y'],smos_ease)}
            ds = xr.Dataset(data_vars = data_vars,
                            coords={'longitude':(['x','y'],ease_lons),
                                    'latitude':(['x','y'],ease_lats)})
            ds.to_netcdf(filepath_out,'w')